In [1]:
import requests
import json
import time
from pathlib import Path
import re
import wikitextparser as wtp
import pycountry
from tqdm.notebook import tqdm

In [3]:
API_URL = "https://marvelcinematicuniverse.fandom.com/api.php"

# Start with a small collection of important MCU pages.
PAGES = [
    "Iron Man",
    "The Incredible Hulk",
    "Iron Man 2",
    "Black Widow (film)",
    "Thor (film)",
    "Captain America: The First Avenger",
    "The Avengers",
    "Iron Man 3",
    "Thor: The Dark World",
    "Captain America: The Winter Soldier",
    "Guardians of the Galaxy (film)",
    "Avengers: Age of Ultron",
    "Ant-Man (film)",
    "Captain America: Civil War",
    "Doctor Strange (film)",
    "Guardians of the Galaxy Vol. 2",
    "Spider-Man: Homecoming",
    "Thor: Ragnarok",
    "Black Panther (film)",
    "Avengers: Infinity War",
    "Ant-Man and the Wasp",
    "Captain Marvel (film)",
    "Avengers: Endgame",
    "Spider-Man: Far From Home",
    "Shang-Chi and the Legend of the Ten Rings",
    "Eternals (film)",
    "Spider-Man: No Way Home",
    "Doctor Strange in the Multiverse of Madness",
    "Thor: Love and Thunder",
    "Black Panther: Wakanda Forever",
    "Ant-Man and the Wasp: Quantumania",
    "Guardians of the Galaxy Vol. 3",
    "The Marvels",
    "Deadpool & Wolverine",
    "Captain America: Brave New World",
    "Thunderbolts*",
    "The Fantastic Four: First Steps",
    "Spider-Man: Brand New Day"
]

OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)

session = requests.Session()

session.headers.update({
    "User-Agent": "MCU-Lore-RAG/1.0"
})

In [3]:
def get_page(title):
    # Be respectful with requests.
    time.sleep(7)

    params = {
        "action": "query",
        "prop": "revisions",
        "titles": title,
        "rvprop": "content|timestamp|ids",
        "rvslots": "main",
        "formatversion": "2",
        "format": "json"
    }

    response = session.get(
        API_URL,
        params=params,
        timeout=30
    )

    #print(f"Downloading: {title}")

    response.raise_for_status()

    data = response.json()

    pages = data.get("query", {}).get("pages", [])

    if not pages:
        return None

    page = pages[0]

    if "missing" in page:
        print(f"[NOT FOUND] {title}")
        return None

    revision = page.get("revisions", [{}])[0]

    return {
        "title": page.get("title"),
        "page_id": page.get("pageid"),
        "timestamp": revision.get("timestamp"),
        "content": revision.get("slots", {})
            .get("main", {})
            .get("content", ""),
        "source": f"https://marvelcinematicuniverse.fandom.com/wiki/{title.replace(' ', '_')}"
    }

In [4]:
def extract_sections(parsed_page, page_title, source):
    data = list()
    links = set()

    for section in parsed_page.sections:
        
        if section.title in ["References", "External Links"]:
            continue

        for link in section.wikilinks:
            if link.title.isdigit():
                continue
            links.add(link.title)

        data.append({
            "page": page_title,
            "section" : section.title,
            "section_level" : section.level,
            "content" : section.plain_text(),
            "source" : "MCU Wiki",
            "discovered_from" : source
        })
        
    return data, sorted(links)
    

In [5]:
LANGUAGE_CODES = {
    "en", "de", "es", "fr", "it", "ja", "ko",
    "pl", "pt", "pt-br", "ru", "tr", "uk", "zh"
}

EXCLUDED_PREFIXES = {
    "file",
    "image",
    "category",
    "template",
    "special",
    "help",
    "media",
    "portal",
    "user",
    "talk",
    "project"
}

COUNTRIES = {
    country.name.lower()
    for country in pycountry.countries
}

def links_filter(links):
    filtered_links = links.copy()
    for link in links:
        title = link.strip()
        # Empty
        if not title:
            filtered_links.remove(link)
            continue

        # Remove leading colon
        title = title.lstrip(":").strip()
        
        # Remove fragment: "Tony Stark#History" → "Tony Stark"
        title = title.split("#")[0].strip()

        if title.lower() in COUNTRIES:
            filtered_links.remove(link)
            continue

        if title[:4].isdigit() and title[-1] == 's' and len(title) == 5:
            filtered_links.remove(link)

        # Pure numbers / years
        if title.isdigit():
            filtered_links.remove(link)

        # Years such as 1970, 2023, etc.
        if re.fullmatch(r"\d{4}", title):
            filtered_links.remove(link)

        # Namespaces / special pages
        if ":" in title:
            prefix = title.split(":", 1)[0].lower().strip()

            if prefix in EXCLUDED_PREFIXES:
                filtered_links.remove(link)

            # Language links
            if prefix in LANGUAGE_CODES:
                filtered_links.remove(link)

    return filtered_links

In [9]:
def process(depth, raw_loc="raw_mcu_pages.json", clean_loc="clean_mcu_pages.json"):
    raw_documents = []
    cleaned_documents = []

    # Pages that have already been/will be processed
    links_done = set(PAGES)

    for _, title in tqdm(enumerate(PAGES, start=1),total=len(PAGES), desc="All Movies Progress"):
        try:
            page = get_page(title)
            if page:

                raw_documents.append(page)

                source = page['source']
                
                parsed_page = wtp.parse(page['content'])

                head_data, links = extract_sections(
                    parsed_page,
                    title,
                    source
                )

                filtered_links = links_filter(links)

                cleaned_documents.append(head_data)

                for _,link in tqdm(enumerate(filtered_links),total=len(filtered_links), desc=f"{title} Progress", position=1):

                    link_title = link.strip()

                    if link_title in links_done:
                        continue

                    sub_page = get_page(link_title)

                    if not sub_page:
                        continue

                    raw_documents.append(sub_page)

                    parsed_subpage = wtp.parse(sub_page['content'])

                    sub_page_data, _ = extract_sections(
                        parsed_subpage,
                        link_title,
                        source
                    )

                    cleaned_documents.append(sub_page_data)

                    # Mark as processed
                    links_done.add(link_title)

        except requests.RequestException as e:
            print(f"[ERROR] {title}: {e}")

    raw_output_file = OUTPUT_DIR / raw_loc
    clean_output_file = OUTPUT_DIR / clean_loc

    with open(raw_output_file, "w", encoding="utf-8") as f:
        json.dump(
            raw_documents,
            f,
            indent=2,
            ensure_ascii=False
        )

    with open(clean_output_file, "w", encoding="utf-8") as f:
        json.dump(
            cleaned_documents,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(f"\nSaved {len(raw_documents)} pages to:")
    print(raw_output_file)

    print(f"\nSaved {len(cleaned_documents)} pages to:")
    print(clean_output_file)

In [5]:
with open(OUTPUT_DIR/"raw_mcu_pages.json", "r", encoding="utf-8") as f:
    data1 = json.load(f)

with open(OUTPUT_DIR/"raw_missing_content.json", "r", encoding="utf-8") as f:
    data2 = json.load(f)

merged_data = data1 + data2

with open(OUTPUT_DIR/"raw_merged.json", "w", encoding="utf-8") as f:
    json.dump(merged_data, f, ensure_ascii=False, indent=2)